<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi-e2/notebooks/gsm8k/E2_Experiment_CMER_Dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Experiment 2: Two-Step Lookahead MEDAL vs. AB-MEDAL on GSM8K with Common-Mask Entropy Reduction (CMER) — Dev (5 Questions)

This notebook evaluates **Experiment 2 (E2)** comparing **MEDAL** and **AB-MEDAL** on GSM8K using the **Two-Step CMER ($R_{\mathrm{CMER}}$)** reward function with `LLaDA-8B-Instruct`.

### Mathematical Formulation:
For parent state $s$, expanded child state $s_i$, and greedy virtual grandchild $g_i^* = s_i \oplus a^*_i$ ($a^*_i = A(s_i)[0]$), reward is averaged strictly over positions $\mathcal{M}(g_i^*)$ that remain masked in the grandchild (so merely unmasking positions does not yield positive reward):
$$
R_{\mathrm{CMER}}(s, g_i^*) = \frac{1}{|\mathcal{M}(g_i^*)|} \sum_{j \in \mathcal{M}(g_i^*)} \left[ h_j(s) - h_j(g_i^*) \right], \qquad h_j(s) = -\sum_v p_\theta(v \mid j, s) \log p_\theta(v \mid j, s)
$$

### Key Algorithmic & Protocol Rules:
- **Search Budget $B \in \{256, 512\}$**: Measured in actual sequence model forward passes (per-sequence NFEs). Caching saves model forwards without consuming budget.
- **Simulation Cap Safeguard ($4B$)**: Capped at $4B$ completed simulations per question to prevent nonterminating or runaway tree traversal loops over cached states. The NFE utilization metric $\frac{\text{actual NFEs}}{B}$ is monitored to verify whether the cap binds.
- **Virtual Grandchildren ($g_i^*$)**: Evaluated and cached in `cached_analyses` to eliminate redundant forwards, but **never attached to the MCTS tree** and never assigned GEN/CONT posteriors.
- **AB-MEDAL Second-Step Selection**: On `GEN`, the first step creates child $s_i$. The second step performs a forced greedy GEN ($a_i^* = A(s_i)[0]$) to construct $g_i^*$ without mutating $s_i$'s GEN/CONT posteriors or visit counts.
- **Position-Wise Entropy Reuse**: `StateAnalysis.entropies` already stores $\mathbf{h}(s)$ from the single forward pass, adding **0 extra NFEs**.
- **Empty Common Masks & Uniform 1-Step CMER Fallback**: When $|\mathcal{M}(g_i^*)| = 0$ (terminal child or grandchild filling the last mask), $R_{\mathrm{CMER}} = 0.0$. When lookahead is unavailable (actionless non-terminal child or NFE boundary fallback), reward uses the 1-step CMER fallback $\frac{1}{|\mathcal{M}(s_i)|}\sum_{j \in \mathcal{M}(s_i)} [h_j(s) - h_j(s_i)]$ (never mixing SRIG/FIG rewards).
- **Exact Checkpoint Boundaries**: Snapshots at $B=256$ use non-mutating hypothetical boundary backups on isolated copies so that the snapshot exactly matches an independent $B=256$ run while the live tree continues seamlessly to $B=512$.


## 1. Verify GPU Hardware
Verify that a CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM) is active.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.')
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024 ** 3)
print(f'GPU: {gpu.name} | Compute Capability: {gpu.major}.{gpu.minor} | Total VRAM: {vram_gb:.2f} GiB')
if vram_gb < 20.0:
    print('Warning: VRAM is below 22 GiB. If OOM occurs, use --batch-size 1.')
else:
    print('VRAM check passed.')


## 2. Repository Setup
Clone or update the GitHub repository on branch `adi-e2`.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/CooL-Legend/CSCI544-DLM.git'
BRANCH = 'adi-e2'
REPO_DIR = Path('/content/CSCI544-DLM') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        token = os.environ.get('GITHUB_TOKEN', '')
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get('GITHUB_TOKEN')
            except Exception:
                pass
        clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
        subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Working directory: {Path.cwd()}')


## 3. Install Dependencies
Install pinned dependencies and install editable package.

In [ ]:
import site
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements/llada-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
    import importlib
    importlib.invalidate_caches()
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
print('Dependencies installed successfully.')


## 4. Run Algorithmic Pre-Flight Unit Tests
Run unit tests to verify 2-step lookahead CMER reward computation, 1-step CMER fallback, exact checkpoint boundaries, and state caching.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/tasks/test_experiment_2.py', '-v'],
    capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Pre-flight E2 unit tests failed!')
print('All E2 algorithmic unit tests passed.')


## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent artifact and model caching.

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/CSCI544-DLM')
OUTPUT_DIR = Path('results/e2_cmer_dev')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = DRIVE_ROOT / 'results/e2_cmer_dev'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cache_dir = DRIVE_ROOT / 'cache/huggingface'
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

print(f'Output directory: {OUTPUT_DIR.resolve()}')


## 6. Experiment Controls & Configuration
Configure 5-question frozen `dev` set, matched search budgets [256, 512], CMER reward mode, and $4B$ simulation cap.

In [ ]:
from dlm_project.tasks.gsm8k.experiment_2 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context
)

STAGE = 'dev'
METHODS = ('medal', 'ab_medal')
BUDGETS = (256, 512)
REWARD_MODE = 'cmer'
REWARD_LAMBDA = 1.0
SIM_CAP_MULTIPLIER = 4
# On NVIDIA L4 (22.4 GiB VRAM), BATCH_SIZE = 2 is recommended for stability;
# BATCH_SIZE = 4 is suited for A100 (40/80 GB) or High-RAM runtimes.
BATCH_SIZE = 2
RESUME = True

ctx = context(
    stage=STAGE,
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    reward_mode=REWARD_MODE,
    reward_lambda=REWARD_LAMBDA,
    sim_cap_multiplier=SIM_CAP_MULTIPLIER,
)
print(f'Stage: {ctx["stage"]} | Questions: {len(ctx["examples"])} | Budgets: {BUDGETS} | Methods: {METHODS} | Reward: {REWARD_MODE} (λ={REWARD_LAMBDA}) | Sim Cap: {SIM_CAP_MULTIPLIER}B | Batch Size: {BATCH_SIZE}')


## 7. Cache & Load LLaDA-8B-Instruct
Stage model to fast local NVMe storage and load `LLaDA-8B-Instruct` model and tokenizer on CUDA in bfloat16.

In [ ]:
import shutil
from pathlib import Path
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.experiment_2 import load_model, prepare

profile = get_profile('llada')
model_name = profile['repo_id'].split('/')[-1]
drive_model_dir = DRIVE_ROOT / 'models' / model_name
runtime_model_dir = Path('/content/models') / model_name

# Fast local NVMe cache in Colab runtime if available on Drive
if IN_COLAB and drive_model_dir.is_dir() and not (runtime_model_dir / 'config.json').is_file():
    print(f'Copying model from Google Drive to local fast runtime storage: {runtime_model_dir}...')
    shutil.copytree(drive_model_dir, runtime_model_dir)

MODEL_PATH = str(runtime_model_dir if runtime_model_dir.is_dir() else (drive_model_dir if drive_model_dir.is_dir() else profile['repo_id']))
print('Using model location:', MODEL_PATH)
print('Loading model and tokenizer on GPU in bfloat16...')
wrapper = load_model(ctx, model_path=MODEL_PATH, device='cuda', local_files_only=Path(MODEL_PATH).is_dir())
prepared = prepare(wrapper, ctx)
print(f'✓ Model loaded: {wrapper.identity["family"]} ({wrapper.identity["repo_id"]} @ {wrapper.identity["revision"]})')
print(f'✓ Prepared {len(prepared)} tokenized evaluation prompts.')


## 8. Execute Experiment 2 (Dev (5 Questions))
Execute MEDAL and AB-MEDAL with the 2-step lookahead CMER reward across budgets 256 and 512.

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_2 import run_e2_stage

print(f'Starting E2 CMER execution on {len(ctx["examples"])} questions for {METHODS} at budgets {BUDGETS}...')
t_start = perf_counter()
q_rows, agg_rows = run_e2_stage(
    wrapper,
    ctx,
    OUTPUT_DIR,
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
total_elapsed = perf_counter() - t_start
print(f'\nE2 Dev execution complete in {total_elapsed / 60:.2f} minutes ({len(q_rows)} checkpoint outcomes).')


## 9. Experiment 2 Results, NFE Utilization & Paired Comparison
Display aggregate accuracy, NFE utilization, simulation cap binding rate, and McNemar exact test at matched budgets.

In [ ]:
import pandas as pd
from IPython.display import display
from scripts.run_e2 import compute_paired_comparison

agg_file = OUTPUT_DIR / 'summaries' / f'e2_{STAGE}_aggregate.csv'
if agg_file.is_file():
    df_agg = pd.read_csv(agg_file)
    print('=== E2 Dev Aggregate Metrics (CMER Reward) ===')
    cols = ['method', 'search_budget', 'sample_count', 'accuracy_percent', 'mean_nfe_utilization', 'mean_search_nfe', 'mean_simulations', 'sim_cap_bind_rate', 'mean_total_nfe', 'mean_selected_depth']
    available_cols = [c for c in cols if c in df_agg.columns]
    display(df_agg[available_cols])

paired = compute_paired_comparison(q_rows, budgets=BUDGETS)
if paired:
    df_paired = pd.DataFrame(paired)
    print('=== Paired Comparison (AB-MEDAL vs. MEDAL at Matched Search Budget) ===')
    display(df_paired[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']])


## 10. Interactive Search Tree & Decision Trace Inspector
Examine candidate shortlists, 2-step CMER reward breakdowns, and lookahead state entropy drops.

In [ ]:
import json

sample_ex_id = prepared[0]['example']['example_id']
chk_dir = OUTPUT_DIR / STAGE / 'ab_medal' / sample_ex_id / 'checkpoints/256'
if (chk_dir / 'metrics.json').is_file():
    met = json.loads((chk_dir / 'metrics.json').read_text())
    print(f'Sample {sample_ex_id} @ Budget 256:')
    print(f'  Selected Node: {met["selected_node_id"]} | Depth: {met["selected_depth"]} | Q: {met["selected_q"]:.4f}')
    print(f'  Exact Match: {met["exact_match"]} | Pred: {met["predicted_answer"]} | Ref: {met["reference_answer"]}')
    print(f'  NFE Utilization: {met.get("nfe_utilization", 1.0)*100:.1f}% | Sims: {met.get("simulations_completed", "N/A")}/{met.get("simulation_cap", "N/A")}')


## 11. E3 Offline Metric Extraction & Publication Figures
Run offline tree allocation, branching dynamics, and structural analysis on saved E2 checkpoints (zero GPU compute), and generate 6 publication figures.

In [ ]:
from dlm_project.tasks.gsm8k.e3_analysis import analyze_e3_stage
from IPython.display import Image, display, Markdown

print('Running E3 Offline Tree Allocation & Topology Analysis on E2 traces...')
e3_q, e3_agg = analyze_e3_stage(
    results_dir=OUTPUT_DIR / STAGE,
    output_dir=OUTPUT_DIR / 'summaries',
    stage=STAGE,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
)
print(f'✓ E3 Analysis Completed successfully: {len(e3_q)} question outcomes evaluated.')

figures_dir = OUTPUT_DIR / 'summaries' / 'e3_figures'
fig_titles = {
    'fig1_nfe_by_depth': 'Figure 1: NFE Allocation by Tree Depth — MEDAL vs AB-MEDAL across budgets',
    'fig2_branching_factors': 'Figure 2: Branching Factor Distribution — Children per expanded parent',
    'fig3_candidate_density': 'Figure 3: Candidate Density vs Depth — Minimum-C availability and selection depth',
    'fig4_root_subtrees': 'Figure 4: Root-Subtree Compute Concentration — Search effort across root branches',
    'fig5_reward_redundancy': 'Figure 5: Reward and Search Redundancy — Negative IG rates and duplicate states',
    'fig6_accuracy_vs_budget': 'Figure 6: Accuracy vs Search Budget — Performance scaling curve',
}

for fig_key, title in fig_titles.items():
    png_path = figures_dir / f'{fig_key}.png'
    if png_path.is_file():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(png_path), width=750))


## 12. Synchronize All Artifacts to Google Drive
Explicitly flush and sync all in-flight buffers to Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Google Drive flushed successfully.')
